In [3]:
# Data handling
import pandas as pd
import numpy as np

# Logistic regression
import statsmodels.api as sm
import statsmodels.formula.api as smf

# Model evaluation tools that may be needed later
from sklearn.metrics import (
    confusion_matrix,
    classification_report,
    accuracy_score
)

# Optional visualization packages
import matplotlib.pyplot as plt
import seaborn as sns

In [4]:
# CSV file
df = pd.read_csv("/data/workspace_files/SBAcase.11.13.17.csv")

# View the first rows
df.head(2)

,Selected,LoanNr_ChkDgt,Name,City,State,Zip,Bank,BankState,NAICS,ApprovalDate,...,ChgOffPrinGr,GrAppv,SBA_Appv,New,RealEstate,Portion,Recession,daysterm,xx,Default
0,0,1004285007,SIMPLEX OFFICE SOLUTIONS,ANAHEIM,CA,92801,CALIFORNIA BANK & TRUST,CA,532420,15074,...,0,30000,15000,0,0,0.5,0,1080,16175.0,0
1,1,1004535010,DREAM HOME REALTY,TORRANCE,CA,90505,CALIFORNIA BANK & TRUST,CA,531210,15130,...,0,30000,15000,0,0,0.5,1,1680,17658.0,0


In [5]:
print("Dataset shape:", df.shape)
print("\nSelected counts:")
print(df["Selected"].value_counts().sort_index())

print("\nDefault counts:")
print(df["Default"].value_counts().sort_index())

Dataset shape: (2102, 35)

Selected counts:
Selected
0    1051
1    1051
Name: count, dtype: int64

Default counts:
Default
0    1416
1     686
Name: count, dtype: int64


In [6]:
model_columns = [
    "Selected",
    "Default",
    "New",
    "RealEstate",
    "DisbursementGross",
    "Portion",
    "Recession"
]

for column in model_columns:
    df[column] = pd.to_numeric(df[column], errors="coerce")

print(df[model_columns].isna().sum())

Selected             0
Default              0
New                  0
RealEstate           0
DisbursementGross    0
Portion              0
Recession            0
dtype: int64


In [7]:
train = df.loc[df["Selected"] == 1].copy()
test = df.loc[df["Selected"] == 0].copy()

print("Training observations:", len(train))
print("Testing observations:", len(test))

Training observations: 1051
Testing observations: 1051


In [8]:
predictors_7a = [
    "New",
    "RealEstate",
    "DisbursementGross",
    "Portion",
    "Recession"
]

X_train_7a = sm.add_constant(
    train[predictors_7a],
    has_constant="add"
)

y_train = train["Default"]

model_7a = sm.GLM(
    y_train,
    X_train_7a,
    family=sm.families.Binomial()
)

result_7a = model_7a.fit()

print(result_7a.summary())

                 Generalized Linear Model Regression Results                  
Dep. Variable:                Default   No. Observations:                 1051
Model:                            GLM   Df Residuals:                     1045
Model Family:                Binomial   Df Model:                            5
Link Function:                  Logit   Scale:                          1.0000
Method:                          IRLS   Log-Likelihood:                -540.85
Date:                Fri, 25 Sep 2026   Deviance:                       1081.7
Time:                        08:36:40   Pearson chi2:                 1.00e+03
No. Iterations:                     6   Pseudo R-squ. (CS):             0.1949
Covariance Type:            nonrobust                                         
                        coef    std err          z      P>|z|      [0.025      0.975]
-------------------------------------------------------------------------------------
const                 1.3537      0.32

In [9]:
table_7a = pd.DataFrame({
    "Estimate": result_7a.params,
    "Standard Error": result_7a.bse,
    "Wald Chi-Square": (result_7a.params / result_7a.bse) ** 2,
    "p-value": result_7a.pvalues
})

table_7a.round(4)

,Estimate,Standard Error,Wald Chi-Square,p-value
const,1.3537,0.3229,17.5729,0.0000
New,-0.0772,0.2101,0.1349,0.7134
RealEstate,-2.0331,0.3636,31.2663,0.0000
DisbursementGross,-0.0000,0.0000,0.9173,0.3382
Portion,-2.8298,0.5594,25.5909,0.0000
Recession,0.4971,0.2413,4.2441,0.0394


In [10]:
predictors_8 = [
    "RealEstate",
    "Portion",
    "Recession"
]

X_train_8 = sm.add_constant(
    train[predictors_8],
    has_constant="add"
)

model_8 = sm.GLM(
    y_train,
    X_train_8,
    family=sm.families.Binomial()
)

result_8 = model_8.fit()

print(result_8.summary())

                 Generalized Linear Model Regression Results                  
Dep. Variable:                Default   No. Observations:                 1051
Model:                            GLM   Df Residuals:                     1047
Model Family:                Binomial   Df Model:                            3
Link Function:                  Logit   Scale:                          1.0000
Method:                          IRLS   Log-Likelihood:                -541.38
Date:                Fri, 25 Sep 2026   Deviance:                       1082.8
Time:                        08:37:12   Pearson chi2:                     989.
No. Iterations:                     6   Pseudo R-squ. (CS):             0.1941
Covariance Type:            nonrobust                                         
                 coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------
const          1.3931      0.322      4.332      0.0

In [11]:
table_8 = pd.DataFrame({
    "Estimate": result_8.params,
    "Standard Error": result_8.bse,
    "Wald Chi-Square": (result_8.params / result_8.bse) ** 2,
    "p-value": result_8.pvalues
})

table_8.round(4)

,Estimate,Standard Error,Wald Chi-Square,p-value
const,1.3931,0.3216,18.7670,0.0000
RealEstate,-2.1282,0.3450,38.0529,0.0000
Portion,-2.9875,0.5393,30.6898,0.0000
Recession,0.5041,0.2412,4.3679,0.0366


In [12]:
X_test_8 = sm.add_constant(
    test[predictors_8],
    has_constant="add"
)

In [13]:
test["PredictedProbability"] = result_8.predict(X_test_8)

test[
    ["Default", "RealEstate", "Portion",
     "Recession", "PredictedProbability"]
].head()

,Default,RealEstate,Portion,Recession,PredictedProbability
0,0,0,0.50,0,0.474852
2,0,0,0.50,0,0.474852
6,1,1,0.75,0,0.048533
7,0,0,0.50,0,0.474852
8,1,0,0.50,0,0.474852


In [14]:
test["PredictedDefault"] = (
    test["PredictedProbability"] >= 0.50
).astype(int)

In [15]:
test["Classification"] = np.where(
    test["PredictedDefault"] == 1,
    "Higher risk",
    "Lower risk"
)

test["ActualStatus"] = np.where(
    test["Default"] == 1,
    "Charged off",
    "Paid in full"
)

In [16]:
table_9 = pd.crosstab(
    test["Classification"],
    test["ActualStatus"]
)

table_9 = table_9.reindex(
    index=["Higher risk", "Lower risk"],
    columns=["Charged off", "Paid in full"]
)

table_9

ActualStatus,Charged off,Paid in full
Classification,,
Higher risk,31,14
Lower risk,324,682


In [17]:
misclassified = (
    test["PredictedDefault"] != test["Default"]
).sum()

misclassification_rate = misclassified / len(test)

print("Number misclassified:", misclassified)
print(f"Misclassification rate: {misclassification_rate:.4f}")
print(f"Misclassification percentage: {misclassification_rate:.2%}")

Number misclassified: 338
Misclassification rate: 0.3216
Misclassification percentage: 32.16%
